# Build model — NEW aggregator

Lightweight model on the **new** trade features (`processed_new`).

- **trade** = the new `Features_To_Use.json` subset (the percent-of-DQ features the fix changes)
- **app** = all columns, **target** = `final_DQ60_m24`
- **train** on Q2'19 rows, **evaluated** on the test rows (experian Q3'19 + eq/tu Q1'20) via `data_split`
- LevelSelection -> FillNA -> XGBoost (depth 3, n_estimators 500)

Output -> `payment_processing_research_data/models/model_new/`.
Run after the `fix_processing/Sample_*` notebooks have produced the samples. Model-engine kernel.

In [ ]:
1+1

In [ ]:
import os, json, importlib
import model_configs
importlib.reload(model_configs)
from model_engine.model_builder.build_model import build_model

VARIANT = 'new'
out_dir = os.path.join(model_configs.MODELS_DIR, f'model_{VARIANT}')
os.makedirs(out_dir, exist_ok=True)
print('variant:', VARIANT, '| output ->', out_dir)

In [ ]:
asset = model_configs.build_asset(VARIANT)



In [ ]:
asset

In [ ]:
# sanity: every table has files before we train
for tbl in ['app', 'trade', 'target']:
    n = len(asset['data'][tbl]['data'])
    print(f'{tbl:7}: {n} files')
    assert n > 0, f'no {tbl} files -- run the fix_processing/Sample_* notebooks first'
print('trade keep_features:', 'ALL' if asset['data']['trade']['io_params']['keep_features'] is None
      else len(asset['data']['trade']['io_params']['keep_features']))
print('data_split:', asset['config']['data_split'])

json.dump(asset, open(os.path.join(out_dir, 'asset.json'), 'w'), indent=2)
print('wrote', os.path.join(out_dir, 'asset.json'))

In [ ]:
import logging, traceback

log_path = os.path.join(out_dir, f'build_model_{VARIANT}.log')
fh = logging.FileHandler(log_path, mode='w')
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s'))
root = logging.getLogger(); root.addHandler(fh); root.setLevel(logging.INFO)

try:
    build_model(asset, out_dir)
    root.info('BUILD SUCCEEDED -> %s', out_dir)
    print('\nDONE -> model artifacts in', out_dir)
    print(sorted(os.listdir(out_dir)))
except Exception as e:
    root.error('BUILD FAILED: %s\n%s', e, traceback.format_exc())
    print('BUILD FAILED -- see log:', log_path)
    raise
finally:
    root.removeHandler(fh); fh.close()
    print('log saved ->', log_path)